# Filter Wheel Smoke Tests

Staged tests for the EvoMachine filter-wheel API. Virtual hardware is the safe default. The physical ASI Tiger movement cells are guarded so that initialisation and inspection do not move the wheel.

In [8]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
import time


def find_repo_root() -> Path:
    """Return the repository root containing the evomachine package."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "evomachine" / "peripherals" / "filterwheel.py").is_file():
            return candidate
    raise RuntimeError("Could not find the EvoMachine repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evomachine.bindings.binding_types import BindingType
from evomachine.peripherals.filterwheel import (
    FilterWheelConfig,
    FilterWheelFactory,
)
from evomachine.peripherals.peripheralcontrollers import (
    PeripheralControllerConfig,
    PeripheralControllerFactory,
    SerialPeripheralControllerConfig,
)
from evomachine.types import FilterWheelType


@dataclass(frozen=True)
class FilterPosition:
    filter_type: FilterWheelType
    position: int


@dataclass(frozen=True, kw_only=True)
class FilterWheelTestSettings:
    binding: BindingType = BindingType.ASI_TIGER
    available_filters: tuple[FilterWheelType, ...] = (
        FilterWheelType.FILTER,
        FilterWheelType.FILTER_465nm,
        FilterWheelType.FILTER_527nm,
        FilterWheelType.FILTER_592nm,
        FilterWheelType.NO_FILTER,
        FilterWheelType.BLOCKING,
    )
    port: str | None = None
    hwid: str | None = None
    card_address: int = 8
    target_filter: FilterWheelType = FilterWheelType.FILTER_592nm
    settle_time_s: float = 1.0
    positions: tuple[FilterPosition, ...] = (
        FilterPosition(FilterWheelType.FILTER, 0),
        FilterPosition(FilterWheelType.FILTER_465nm, 1),
        FilterPosition(FilterWheelType.FILTER_527nm, 2),
        FilterPosition(FilterWheelType.FILTER_592nm, 3),
        FilterPosition(FilterWheelType.NO_FILTER, 4),
        FilterPosition(FilterWheelType.BLOCKING, 5),
    )


VIRTUAL_SETTINGS = FilterWheelTestSettings()
TIGER_SETTINGS = FilterWheelTestSettings(
    binding=BindingType.ASI_TIGER,
    hwid="USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3",
)

SETTINGS = TIGER_SETTINGS  # Change to TIGER_SETTINGS for physical hardware.
RUN_FILTER_MOVE = True  # Change deliberately only after checking the mapping.
SETTINGS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


FilterWheelTestSettings(binding=<BindingType.ASI_TIGER: 2>, available_filters=(<FilterWheelType.FILTER: 0>, <FilterWheelType.FILTER_465nm: 1>, <FilterWheelType.FILTER_527nm: 2>, <FilterWheelType.FILTER_592nm: 3>, <FilterWheelType.NO_FILTER: 4>, <FilterWheelType.BLOCKING: 5>), port=None, hwid='USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3', card_address=8, target_filter=<FilterWheelType.FILTER_592nm: 3>, settle_time_s=1.0, positions=(FilterPosition(filter_type=<FilterWheelType.FILTER: 0>, position=0), FilterPosition(filter_type=<FilterWheelType.FILTER_465nm: 1>, position=1), FilterPosition(filter_type=<FilterWheelType.FILTER_527nm: 2>, position=2), FilterPosition(filter_type=<FilterWheelType.FILTER_592nm: 3>, position=3), FilterPosition(filter_type=<FilterWheelType.NO_FILTER: 4>, position=4), FilterPosition(filter_type=<FilterWheelType.BLOCKING: 5>, position=5)))

## Inspect serial ports

Use this to confirm the ASI Tiger `port` or `hwid`. Exactly one is required for physical hardware.

In [24]:
try:
    from serial.tools import list_ports
except ImportError as error:
    raise RuntimeError("pyserial is required to inspect serial ports.") from error

[
    {"device": port.device, "description": port.description, "hwid": port.hwid}
    for port in list_ports.comports()
]

[{'device': '/dev/ttyS1', 'description': 'ttyS1', 'hwid': 'PNP0501'},
 {'device': '/dev/ttyS0', 'description': 'ttyS0', 'hwid': 'PNP0501'},
 {'device': '/dev/ttyUSB0',
  'description': 'CP2102 USB to UART Bridge Controller - CP2102 USB to UART Bridge Controller',
  'hwid': 'USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3'},
 {'device': '/dev/ttyACM4',
  'description': 'USB Serial',
  'hwid': 'USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0'}]

## Review the logical-to-physical position mapping

These integer positions are sent directly to the ASI Tiger. Confirm them against the installed wheel before enabling movement.

In [25]:
[
    {"filter": item.filter_type.name, "position": item.position}
    for item in SETTINGS.positions
]

[{'filter': 'FILTER', 'position': 0},
 {'filter': 'FILTER_465nm', 'position': 1},
 {'filter': 'FILTER_527nm', 'position': 2},
 {'filter': 'FILTER_592nm', 'position': 3},
 {'filter': 'NO_FILTER', 'position': 4},
 {'filter': 'BLOCKING', 'position': 5}]

## Create and initialise the filter wheel

ASI Tiger initialisation checks the controller but does not command a wheel position. Its current filter is reported as `UNKNOWN` because physical readback is not implemented yet.

In [9]:
previous_wheel = globals().get("filter_wheel")
if previous_wheel is not None and previous_wheel.is_initialised():
    previous_wheel.finalise()
previous_controller = globals().get("filter_controller")
if previous_controller is not None and previous_controller.is_initialised():
    previous_controller.shutdown()

if SETTINGS.binding == BindingType.VIRTUAL:
    controller_config = PeripheralControllerConfig(binding=BindingType.VIRTUAL)
    filter_controller = PeripheralControllerFactory.create(controller_config)
elif SETTINGS.binding == BindingType.ASI_TIGER:
    controller_config = SerialPeripheralControllerConfig(
        binding=BindingType.ASI_TIGER,
        port=SETTINGS.port,
        hwid=SETTINGS.hwid,
    )
    filter_controller = PeripheralControllerFactory.create(
        controller_config,
        card_address_filter_wheel=SETTINGS.card_address,
    )
else:
    raise ValueError(f"Unsupported filter-wheel binding: {SETTINGS.binding}")

position_mapping = {item.filter_type: item.position for item in SETTINGS.positions}
wheel_config = FilterWheelConfig(
    binding=SETTINGS.binding,
    available_filters=list(SETTINGS.available_filters),
)
if SETTINGS.binding == BindingType.VIRTUAL:
    filter_wheel = FilterWheelFactory.create(
        wheel_config,
        peripheral_controllers=filter_controller,
        current_filter_type=FilterWheelType.UNKNOWN,
    )
else:
    filter_wheel = FilterWheelFactory.create(
        wheel_config,
        peripheral_controllers=filter_controller,
        filter_wheel_settings=position_mapping,
    )
filter_wheel.initialise()

{
    "name": filter_wheel.name,
    "binding": SETTINGS.binding,
    "controller_initialised": filter_controller.is_initialised(),
    "wheel_initialised": filter_wheel.is_initialised(),
    "wheel_alive": filter_wheel.is_alive(),
    "available_filters": filter_wheel.get_available_filters(),
    "current_filter": filter_wheel.get_filter_wheel(),
}

2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Connecting to /dev/ttyUSB0 at 115200 baud
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:27 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.initialise: initialising ASI Tiger Filter Wheel with force=False.
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:27 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:27 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.initialise: ASI Tiger Filter Wheel initialised at UN

{'name': 'ASI Tiger Filter Wheel',
 'binding': <BindingType.ASI_TIGER: 2>,
 'controller_initialised': True,
 'wheel_initialised': True,
 'wheel_alive': True,
 'available_filters': [<FilterWheelType.FILTER: 0>,
  <FilterWheelType.FILTER_465nm: 1>,
  <FilterWheelType.FILTER_527nm: 2>,
  <FilterWheelType.FILTER_592nm: 3>,
  <FilterWheelType.NO_FILTER: 4>,
  <FilterWheelType.BLOCKING: 5>],
 'current_filter': <FilterWheelType.UNKNOWN: -1>}

## Move to one filter

This movement is blocked until `RUN_FILTER_MOVE` is set to `True`.

In [4]:
if not RUN_FILTER_MOVE:
    raise RuntimeError("Set RUN_FILTER_MOVE = True after checking the physical position mapping.")
if SETTINGS.target_filter not in filter_wheel.get_available_filters():
    raise ValueError(f"{SETTINGS.target_filter} is not configured as an available filter.")

previous_filter = filter_wheel.get_filter_wheel()
filter_wheel.set_filter_wheel(SETTINGS.target_filter)
time.sleep(SETTINGS.settle_time_s)

{
    "previous_filter": previous_filter,
    "commanded_filter": SETTINGS.target_filter,
    "cached_filter": filter_wheel.get_filter_wheel(),
}

2026-07-02 16:45:47 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:45:47 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:45:47 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.set_filter_wheel: setting ASI Tiger Filter Wheel to FILTER_592nm with force=False.
2026-07-02 16:45:47 - DEBUG - asitiger.serialconnection - Sending data: b'8MP 3\r'
2026-07-02 16:45:47 - DEBUG - asitiger.serialconnection - Received: b'3MP 3  3\n'


{'previous_filter': <FilterWheelType.UNKNOWN: -1>,
 'commanded_filter': <FilterWheelType.FILTER_592nm: 3>,
 'cached_filter': <FilterWheelType.FILTER_592nm: 3>}

## Exercise every configured position

This commands each configured position once. Skip it unless the full mapping and mechanical path are known to be safe.

In [10]:
if not RUN_FILTER_MOVE:
    raise RuntimeError("Set RUN_FILTER_MOVE = True after checking the physical position mapping.")

movement_log = []
for filter_type in filter_wheel.get_available_filters():
    print(f"Moving to {filter_type.name}")
    filter_wheel.set_filter_wheel(filter_type, force=True)
    time.sleep(SETTINGS.settle_time_s)
    movement_log.append(filter_wheel.get_filter_wheel())
movement_log

2026-07-02 16:47:32 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:32 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:32 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.set_filter_wheel: setting ASI Tiger Filter Wheel to FILTER with force=True.
2026-07-02 16:47:32 - DEBUG - asitiger.serialconnection - Sending data: b'8MP 0\r'
2026-07-02 16:47:32 - DEBUG - asitiger.serialconnection - Received: b'0MP 0  0\n'


Moving to FILTER


2026-07-02 16:47:33 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:33 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:33 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.set_filter_wheel: setting ASI Tiger Filter Wheel to FILTER_465nm with force=True.
2026-07-02 16:47:33 - DEBUG - asitiger.serialconnection - Sending data: b'8MP 1\r'
2026-07-02 16:47:33 - DEBUG - asitiger.serialconnection - Received: b'1MP 1  1\n'


Moving to FILTER_465nm


2026-07-02 16:47:34 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:34 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:34 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.set_filter_wheel: setting ASI Tiger Filter Wheel to FILTER_527nm with force=True.
2026-07-02 16:47:34 - DEBUG - asitiger.serialconnection - Sending data: b'8MP 2\r'
2026-07-02 16:47:34 - DEBUG - asitiger.serialconnection - Received: b'2MP 2  2\n'


...

2026-07-02 16:47:36 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:36 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:36 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.set_filter_wheel: setting ASI Tiger Filter Wheel to NO_FILTER with force=True.
2026-07-02 16:47:36 - DEBUG - asitiger.serialconnection - Sending data: b'8MP 4\r'
2026-07-02 16:47:36 - DEBUG - asitiger.serialconnection - Received: b'4MP 4  4\n'


Moving to NO_FILTER


2026-07-02 16:47:37 - DEBUG - asitiger.serialconnection - Sending data: b'/\r'
2026-07-02 16:47:37 - DEBUG - asitiger.serialconnection - Received: b'N\r\n'
2026-07-02 16:47:37 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.set_filter_wheel: setting ASI Tiger Filter Wheel to BLOCKING with force=True.
2026-07-02 16:47:37 - DEBUG - asitiger.serialconnection - Sending data: b'8MP 5\r'
2026-07-02 16:47:37 - DEBUG - asitiger.serialconnection - Received: b'5MP 5  5\n'


Moving to BLOCKING


[<FilterWheelType.FILTER: 0>,
 <FilterWheelType.FILTER_465nm: 1>,
 <FilterWheelType.FILTER_527nm: 2>,
 <FilterWheelType.FILTER_592nm: 3>,
 <FilterWheelType.NO_FILTER: 4>,
 <FilterWheelType.BLOCKING: 5>]

## Cleanup

Cleanup releases the controller without commanding another position.

In [11]:
if "filter_wheel" in globals() and filter_wheel is not None:
    if filter_wheel.is_initialised():
        filter_wheel.stop()
        filter_wheel.finalise()

if "filter_controller" in globals() and filter_controller is not None:
    if filter_controller.is_initialised():
        filter_controller.shutdown()

{
    "wheel_initialised": filter_wheel.is_initialised(),
    "controller_initialised": filter_controller.is_initialised(),
}

2026-07-02 16:47:57 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.stop: ASI Tiger Filter Wheel has no generic stop command; skipping.
2026-07-02 16:47:57 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.finalise: finalising ASI Tiger Filter Wheel with force=False.
2026-07-02 16:47:57 - DEBUG - asitiger.serialconnection - Sending data: b'\\\r'
2026-07-02 16:47:57 - DEBUG - asitiger.serialconnection - Received: b':A\r\n'
2026-07-02 16:47:57 - DEBUG - asitiger.serialconnection - Disconnecting from serial port...
2026-07-02 16:47:57 - DEBUG - asitiger.serialconnection - Disconnected


{'wheel_initialised': False, 'controller_initialised': False}